# Student Performance Prediction: Model Comparison

**Research question:** How accurately can final Mathematics grade (`G3`) be predicted from selected non-grade characteristics, without using earlier grades `G1` and `G2`?

This notebook follows the plan in `methodology.md`. It cleans and checks the UCI Mathematics dataset, engineers two features, trains multiple linear regression and Ridge regression, and evaluates both on the same held-out test observations. The dataset is downloaded from the official UCI archive if a local copy is not available.

In [1]:
from pathlib import Path
from urllib.request import urlopen
from zipfile import ZipFile
from io import BytesIO
import math
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

RANDOM_SEED = 42
DATA_DIR = Path("data")
LOCAL_CSV = DATA_DIR / "student-mat.csv"
UCI_ZIP_URL = "https://archive.ics.uci.edu/static/public/320/student+performance.zip"
print("Imports complete. Random seed:", RANDOM_SEED)


Imports complete. Random seed: 42

## 1. Load the data
The raw CSV is semicolon-delimited. If it is not present locally, the notebook downloads the official UCI ZIP archive and reads the Mathematics file directly from it.

In [2]:
if LOCAL_CSV.exists():
    df = pd.read_csv(LOCAL_CSV, sep=";")
    data_source = str(LOCAL_CSV)
else:
    with urlopen(UCI_ZIP_URL, timeout=30) as response:
        archive_bytes = response.read()
    with ZipFile(BytesIO(archive_bytes)) as archive:
        csv_member = next(name for name in archive.namelist()
                          if name.lower().endswith("student-mat.csv"))
        with archive.open(csv_member) as csv_file:
            df = pd.read_csv(csv_file, sep=";")
    data_source = "UCI Student Performance ZIP archive"

print("Source:", data_source)
print("Dataset dimensions (rows, columns):", df.shape)
print("First five columns:", list(df.columns[:5]))
print("Target column present:", "G3" in df.columns)


Source: UCI Student Performance ZIP archive
Dataset dimensions (rows, columns): (395, 33)
First five columns: ['school', 'sex', 'age', 'address', 'famsize']
Target column present: True

## 2. Data quality checks and cleaning
Check duplicates, missingness, data types, and the target range before fitting models. Numeric conversion is explicit for all planned predictors and the target. Rows with missing values in the analysis columns are removed; the checks below report the resulting dataset.

In [3]:
print("Duplicate rows:", int(df.duplicated().sum()))
print("Missing values in raw data:", int(df.isna().sum().sum()))
print("G3 minimum / maximum:", int(df["G3"].min()), int(df["G3"].max()))

base_features = ["age", "studytime", "absences", "famrel",
                 "freetime", "goout", "health"]
numeric_columns = base_features + ["failures", "G3"]
for column in numeric_columns:
    df[column] = pd.to_numeric(df[column], errors="coerce")

analysis_columns = numeric_columns
analysis_df = df[analysis_columns].copy()
analysis_df = analysis_df.drop_duplicates()
analysis_df = analysis_df.dropna(subset=analysis_columns)
analysis_df = analysis_df.loc[analysis_df["G3"].between(0, 20)].copy()

print("Rows after cleaning:", len(analysis_df))
print("Missing values after cleaning:", int(analysis_df.isna().sum().sum()))
print("G3 remains within 0–20:", bool(analysis_df["G3"].between(0, 20).all()))


Duplicate rows: 0
Missing values in raw data: 0
G3 minimum / maximum: 0 20
Rows after cleaning: 395
Missing values after cleaning: 0
G3 remains within 0–20: True

## 3. Feature engineering
Create a binary indicator for whether the student has any previous class failure and another indicator for a higher study-time category (categories 3 or 4). The raw `failures` field is used to construct the indicator and is not also included, avoiding a redundant predictor. Earlier grades `G1` and `G2` are deliberately excluded.

In [4]:
analysis_df["prior_failure_flag"] = (analysis_df["failures"] > 0).astype(int)
analysis_df["studytime_high"] = (analysis_df["studytime"] >= 3).astype(int)

feature_columns = base_features + ["prior_failure_flag", "studytime_high"]
target_column = "G3"
X = analysis_df[feature_columns].astype(float)
y = analysis_df[target_column].astype(float)

print("Feature columns:", feature_columns)
print("Target:", target_column)
print("Rows / features:", X.shape)
print("Earlier grades included:", False)
display(X.head())


Feature columns: ['age', 'studytime', 'absences', 'famrel', 'freetime', 'goout', 'health', 'prior_failure_flag', 'studytime_high']
Target: G3
Rows / features: (395, 9)
Earlier grades included: False


   age  studytime  absences  famrel  freetime  goout  health  prior_failure_flag  studytime_high
0  18.0        2.0       6.0     4.0       3.0    4.0     3.0                   0               0
1  17.0        2.0       4.0     5.0       3.0    3.0     3.0                   0               0
2  15.0        2.0      10.0     4.0       3.0    2.0     3.0                   0               0
3  15.0        3.0       2.0     3.0       2.0    2.0     5.0                   0               1
4  16.0        2.0       4.0     4.0       3.0    2.0     5.0                   0               0

## 4. Reproducible train/test split
A fixed-seed Fisher–Yates shuffle creates an 80/20 holdout split. The same indices are used for both models, so the comparison is paired on the same test observations. A custom deterministic shuffle is used so the split logic is transparent and reproducible.

In [5]:
def deterministic_split_indices(n, seed=42, test_fraction=0.20):
    indices = list(range(n))
    state = seed
    for i in range(n - 1, 0, -1):
        state = (1664525 * state + 1013904223) & 0xFFFFFFFF
        j = int((state / 4294967296) * (i + 1))
        indices[i], indices[j] = indices[j], indices[i]
    n_test = math.ceil(n * test_fraction)
    return indices[n_test:], indices[:n_test]

train_indices, test_indices = deterministic_split_indices(len(X), RANDOM_SEED)
X_train, X_test = X.iloc[train_indices], X.iloc[test_indices]
y_train, y_test = y.iloc[train_indices], y.iloc[test_indices]

print(f"Training observations: {len(X_train)} ({len(X_train)/len(X):.1%})")
print(f"Test observations: {len(X_test)} ({len(X_test)/len(X):.1%})")
print("Split seed:", RANDOM_SEED)


Training observations: 316 (80.0%)
Test observations: 79 (20.0%)
Split seed: 42

## 5. Train two regression models
**Multiple Linear Regression** is the interpretable baseline. **Ridge Regression** adds L2 regularization; its numeric predictors are standardized inside a pipeline, with the scaler fitted only on the training data.

In [6]:
models = {
    "Multiple Linear Regression": LinearRegression(),
    "Ridge Regression (alpha=1)": make_pipeline(
        StandardScaler(),
        Ridge(alpha=1.0)
    )
}

predictions = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    predictions[name] = model.predict(X_test)

print("Models fitted:", ", ".join(models.keys()))


Models fitted: Multiple Linear Regression, Ridge Regression (alpha=1)

## 6. Evaluate on the held-out test set
RMSE and MAE are reported in grade points (lower is better). R² compares the model against a mean-based reference (higher is generally better; negative values are possible on test data).

In [7]:
results = []
for name, y_pred in predictions.items():
    results.append({
        "Model": name,
        "RMSE": mean_squared_error(y_test, y_pred) ** 0.5,
        "MAE": mean_absolute_error(y_test, y_pred),
        "R2": r2_score(y_test, y_pred)
    })

results_df = pd.DataFrame(results).set_index("Model")
display(results_df.round(3))


,RMSE,MAE,R2
Multiple Linear Regression,4.266,3.311,0.064
Ridge Regression (alpha=1),4.266,3.309,0.064


## 7. Findings and next steps

In [8]:
best_by_rmse = results_df["RMSE"].idxmin()
best_by_mae = results_df["MAE"].idxmin()
findings = (
    f"On the fixed 80/20 holdout split, both models produced an RMSE of about "
    f"{results_df['RMSE'].min():.2f} grade points and an R² of about "
    f"{results_df['R2'].max():.3f}, so the selected non-grade features explain "
    f"only a small share of test-set variation. Ridge regression had the slightly "
    f"lower MAE ({results_df.loc['Ridge Regression (alpha=1)', 'MAE']:.2f}) than "
    f"ordinary linear regression ({results_df.loc['Multiple Linear Regression', 'MAE']:.2f}), "
    f"but the difference is very small. These results suggest that the selected "
    f"demographic, study-habit, and attendance-related features alone have limited "
    f"predictive power for final Mathematics grades in this split. The result does "
    f"not show that any feature causes higher or lower grades, and the performance "
    f"may change under a different split. With more time, I would use repeated "
    f"cross-validation, inspect residuals and influential observations, compare "
    f"additional models, and evaluate a separate scenario that includes G1 and G2."
)
print(findings)


On the fixed 80/20 holdout split, both models produced an RMSE of about 4.27 grade points and an R² of about 0.064, so the selected non-grade features explain only a small share of test-set variation. Ridge regression had the slightly lower MAE (3.31) than ordinary linear regression (3.31), but the difference is very small. These results suggest that the selected demographic, study-habit, and attendance-related features alone have limited predictive power for final Mathematics grades in this split. The result does not show that any feature causes higher or lower grades, and the performance may change under a different split. With more time, I would use repeated cross-validation, inspect residuals and influential observations, compare additional models, and evaluate a separate scenario that includes G1 and G2.

## References
- Cortez, P., & Silva, A. M. G. (2008). *Using data mining to predict secondary school student performance*. https://archive.ics.uci.edu/dataset/320/student+performance
- Zhang, Y., Yun, Y., An, R., Cui, J., Dai, H., & Shang, X. (2021). *Educational Data Mining Techniques for Student Performance Prediction: Method Review and Comparison Analysis*. https://doi.org/10.3389/fpsyg.2021.698490
- UCI Machine Learning Repository. *Student Performance*. https://doi.org/10.24432/C5TG7T

**Reproducibility note:** Saved outputs provide a visible run summary. Re-run all cells in an environment with internet access (or with `data/student-mat.csv` present) to regenerate the results.